# 09 - Compare Systems and Make Report Tables

Builds report-ready tables for Base RAG vs Fine-tuned RAG, creates fine-tuned error analysis, and summarizes key experiment artifacts.

In [ ]:
from pathlib import Path
import sys
import importlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; using local filesystem paths.')

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path = [str(DRIVE_ROOT)] + [p for p in sys.path if p != str(DRIVE_ROOT)]
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
importlib.invalidate_caches()

paths = {
    'base_summary': DRIVE_ROOT / 'outputs/generation_eval/base_rag_eval_summary_v1.json',
    'finetuned_summary': DRIVE_ROOT / 'outputs/generation_eval/finetuned_rag_eval_summary_v1.json',
    'finetuned_eval': DRIVE_ROOT / 'outputs/generation_eval/finetuned_rag_eval_v1.csv',
    'normalization_report': DRIVE_ROOT / 'reports/normalization_report_v3.json',
    'leakage_report': DRIVE_ROOT / 'reports/leakage_report.json',
    'external_leakage_report': DRIVE_ROOT / 'reports/leakage_report_external.json',
}

for name, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f'{name}: {path}')

paths

In [ ]:
from src.error_analysis import make_error_analysis
from src.experiment_summary import build_experiment_summary, compare_generation_systems

comparison_df = compare_generation_systems(
    base_summary_json=paths['base_summary'],
    finetuned_summary_json=paths['finetuned_summary'],
    output_csv=DRIVE_ROOT / 'reports/base_vs_finetuned_generation.csv',
    output_markdown=DRIVE_ROOT / 'reports/base_vs_finetuned_generation.md',
)

experiment_df = build_experiment_summary(
    normalization_report_json=paths['normalization_report'],
    leakage_report_json=paths['leakage_report'],
    external_leakage_report_json=paths['external_leakage_report'],
    base_summary_json=paths['base_summary'],
    finetuned_summary_json=paths['finetuned_summary'],
    output_csv=DRIVE_ROOT / 'reports/experiment_summary.csv',
)

finetuned_errors = make_error_analysis(
    eval_csv=paths['finetuned_eval'],
    output_csv=DRIVE_ROOT / 'reports/error_analysis_finetuned_rag.csv',
    per_type_limit=30,
)

comparison_df

In [ ]:
import json

with paths['finetuned_summary'].open('r', encoding='utf-8') as f:
    finetuned_summary = json.load(f)

print('Fine-tuned full error type counts:')
print(json.dumps(finetuned_summary.get('error_type_counts', {}), ensure_ascii=False, indent=2))

print('\nFine-tuned sampled error-analysis rows, capped by per_type_limit=30:')
print(finetuned_errors['error_type_auto'].value_counts().to_string())

print('\nExperiment summary:')
display(experiment_df)

print('\nGenerated report files:')
for rel in [
    'reports/base_vs_finetuned_generation.csv',
    'reports/base_vs_finetuned_generation.md',
    'reports/experiment_summary.csv',
    'reports/error_analysis_finetuned_rag.csv',
]:
    path = DRIVE_ROOT / rel
    print(path, path.exists(), round(path.stat().st_size / 1024, 1), 'KB')

Interpretation note: if Fine-tuned RAG improves `citation_present` but lowers `citation_gold_match` and `grounded_citation_score`, report it as a style/citation-format improvement that did not improve official article grounding. This motivates optimized retrieval and cleaner official-context fine-tuning data.